# geodesiq in JupyterLite: smoke test

Checks that geodesiq and its dependencies work in this browser. Run all cells (*Run → Run All Cells*). Every check runs on its own and prints `PASS` or `FAIL` with its wall time, so one failure does not hide the others. The last cell prints a summary table and raises an error if any check failed; CI runs this notebook on every build of the site.

What is tested, in order of risk:

1. The WebAssembly environment: versions of NumPy, SciPy (≥ 1.15 for `scipy.differentiate`), QuTiP, Matplotlib.
2. `threadpoolctl`, which geodesiq imports at module level and uses in `limit_blas_threads`.
3. The full pipeline: affine and callable `ControlModel`, the diabatic variant, `PulseControl`, and `Dynamics` (`sesolve`, `mesolve`, propagator, `decompose_hamiltonian`, custom pulses).
4. Plotting, file export, `tqdm`, `joblib`.
5. Speed: a reduced Landau–Zener fidelity sweep, extrapolated to the size used in `examples/landau_zener.ipynb`.

In [ ]:
import sys
import time
import traceback

RESULTS = []


def check(name):
    """Run the decorated function at once, record PASS/FAIL, its wall time and its return value."""

    def run(fn):
        t0 = time.perf_counter()
        try:
            out, status = fn(), "PASS"
        except Exception as exc:
            out, status = f"{type(exc).__name__}: {exc}", "FAIL"
            traceback.print_exc()
        dt = time.perf_counter() - t0
        RESULTS.append((name, status, dt, out))
        print(f"[{status}] {name} ({dt:.2f} s): {out}")
        return fn

    return run


print(sys.platform, sys.version)

## 1. Environment

In [ ]:
@check("import numpy, scipy, matplotlib, qutip")
def _():
    import matplotlib
    import numpy
    import qutip
    import scipy

    return {m.__name__: m.__version__ for m in (numpy, scipy, matplotlib, qutip)}


@check("scipy.differentiate.jacobian")
def _():
    import numpy as np
    from scipy.differentiate import jacobian

    res = jacobian(lambda x: np.sin(x), np.array([0.3]))
    return f"d/dx sin(0.3) = {res.df.item():.12f} (exact {np.cos(0.3):.12f})"

## 2. threadpoolctl and BLAS thread limiting

In [ ]:
@check("threadpoolctl.threadpool_info()")
def _():
    import threadpoolctl

    return threadpoolctl.__version__, threadpoolctl.threadpool_info()


@check("import geodesiq")
def _():
    import geodesiq

    return geodesiq.__version__


@check("geodesiq._utils.limit_blas_threads")
def _():
    import numpy as np
    from geodesiq._utils import limit_blas_threads

    with limit_blas_threads(4):
        w = np.linalg.eigvalsh(np.diag([1.0, 2.0, 3.0, 4.0]))
    return w

## 3. Pipeline: Landau–Zener model

In [ ]:
import numpy as np
import qutip as qt

import geodesiq as gq


def ham(z, x):
    return np.array([[z, x], [x, -z]])


@check("affine ControlModel: solve_problem + synthesize_pulse")
def _():
    global model, pulse
    model = gq.ControlModel(H_c=ham(1, 0), H_d=ham(0, 1))
    model.set_control(control_name="z", pulse_initial=-10, pulse_final=10, initial_state=0, alpha=2, beta=2)
    model.solve_problem(pulse_accuracy=1000)
    pulse = model.synthesize_pulse(duration=10)
    return f"{len(pulse.pulse)} samples, z(0) = {pulse.pulse[0]:.4g}, z(tf) = {pulse.pulse[-1]:.4g}"


@check("callable ControlModel (finite differences), matches affine")
def _():
    global model_callable
    model_callable = gq.ControlModel(ham)
    model_callable.set_parameters(x=1)
    model_callable.set_control(control_name="z", pulse_initial=-10, pulse_final=10, initial_state=0, alpha=2, beta=2)
    model_callable.solve_problem(pulse_accuracy=1000)
    diff = np.max(np.abs(model_callable.synthesize_pulse(duration=10).pulse - pulse.pulse))
    assert diff < 1e-6, diff
    return f"max |callable - affine| = {diff:.2e}"


@check("callable ControlModel on a small grid (scipy.differentiate path)")
def _():
    m = gq.ControlModel(ham)
    m.set_parameters(x=1)
    m.set_control(control_name="z", pulse_initial=-10, pulse_final=10, initial_state=0, alpha=2, beta=2, num_steps=17)
    m.solve_problem()
    return f"{len(m.synthesize_pulse(duration=10).pulse)} samples"


@check("diabatic variant (final_state != initial_state)")
def _():
    m = gq.ControlModel(H_c=ham(1, 0), H_d=ham(0, 1))
    m.set_control(
        control_name="z", pulse_initial=-10, pulse_final=10, initial_state=0, final_state=1, alpha=2, beta=2,
        dia_alpha=1, dia_beta=1,
    )
    m.solve_problem()
    return f"{len(m.synthesize_pulse(duration=10).pulse)} samples"

In [ ]:
@check("Dynamics: closed system (sesolve)")
def _():
    global dynamics
    dynamics = gq.Dynamics(model, duration=100)
    return np.ravel(dynamics.state_fidelity())[-1]


@check("Dynamics: dephasing (mesolve)")
def _():
    return np.ravel(dynamics.state_fidelity(c_ops=[np.sqrt(1 / 2) * qt.sigmaz()]))[-1]


@check("Dynamics: average gate fidelity (propagator)")
def _():
    return np.ravel(gq.Dynamics(model, duration=50).average_gate_fidelity(target_gate=qt.sigmax()))[-1]


@check("Dynamics: callable model (decompose_hamiltonian)")
def _():
    return np.ravel(gq.Dynamics(model_callable, duration=100).state_fidelity())[-1]


@check("Dynamics: custom filtered pulse (scipy.signal)")
def _():
    times, filtered = model.synthesize_pulse(duration=100).filtered_pulse(cutoff_freq=0.2)
    return np.ravel(gq.Dynamics(model, times=times, pulse=filtered).state_fidelity())[-1]

## 4. Plotting, export, tqdm, joblib

Three figures should appear below this cell.

In [ ]:
import matplotlib.pyplot as plt


@check("plots (eigenvalues, pulse, populations)")
def _():
    model.plot_eigenvalues()
    pulse.plot_pulse()
    dynamics.plot_populations()
    return "figures created"


plt.show()

In [ ]:
@check("fourier_spectrum (scipy.signal.ShortTimeFFT)")
def _():
    f, t, s = pulse.fourier_spectrum()
    return f"spectrogram {s.shape}"


@check("export_pulse to the browser file system")
def _():
    return str(pulse.export_pulse("smoke_test_pulse", "npz", overwrite=True))


@check("tqdm.auto")
def _():
    from tqdm.auto import tqdm

    return sum(tqdm(range(100)))


@check("joblib.Parallel(n_jobs=2)")
def _():
    from joblib import Parallel, delayed

    return Parallel(n_jobs=2)(delayed(pow)(i, 2) for i in range(4))

In [ ]:
gq.about()

## 5. Speed

`examples/landau_zener.ipynb` sweeps 2 × 1000 durations. Here 20 are timed and the full sweep is extrapolated. Run the same cell in a local Jupyter to get the reference.

In [ ]:
@check("speed: 20 durations of the Landau–Zener sweep")
def _():
    t0 = time.perf_counter()
    for duration in np.linspace(0.01, 10, 20):
        gq.Dynamics(duration=duration, model=model).state_fidelity()
    per_point = (time.perf_counter() - t0) / 20
    return f"{1e3 * per_point:.1f} ms per duration, full sweep (2 x 1000) ~ {2000 * per_point:.0f} s"

## Summary

In [ ]:
n_fail = sum(status == "FAIL" for _, status, _, _ in RESULTS)
print(f"{len(RESULTS) - n_fail}/{len(RESULTS)} passed on {sys.platform}, Python {sys.version.split()[0]}\n")
for name, status, dt, out in RESULTS:
    print(f"{status}  {dt:7.2f} s  {name}" + (f"\n      {out}" if status == "FAIL" else ""))
if n_fail:
    raise RuntimeError(f"{n_fail} smoke-test check(s) failed")